In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

pd.set_option("display.max_columns", None)

In [5]:
# Load dataset
df = pd.read_csv("../data/online_retail_customer_churn.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (789, 9)


,recency,frequency,monetary,avg_spend_per_purchase,customer_value_score,recency_frequency_ratio,monetary_log,Numeric_Mean,recency_x_frequency
0,5,22,5892.58,256.199130,129636.76,0.217391,8.681619,19403.062592,110
1,13,77,9025.47,115.711154,694961.19,0.166667,9.107917,100600.235105,1001
2,13,71,618.83,8.594861,43936.93,0.180556,6.429445,6379.280695,923
3,3,33,9110.30,267.950000,300639.90,0.088235,9.117271,44294.765072,99
4,15,43,5390.88,122.520000,231807.84,0.340909,8.592649,33912.596223,645


In [6]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 789 entries, 0 to 788
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   recency                  789 non-null    int64  
 1   frequency                789 non-null    int64  
 2   monetary                 789 non-null    float64
 3   avg_spend_per_purchase   789 non-null    float64
 4   customer_value_score     789 non-null    float64
 5   recency_frequency_ratio  789 non-null    float64
 6   monetary_log             789 non-null    float64
 7   Numeric_Mean             789 non-null    float64
 8   recency_x_frequency      789 non-null    int64  
dtypes: float64(6), int64(3)
memory usage: 55.6 KB


,recency,frequency,monetary,avg_spend_per_purchase,customer_value_score,recency_frequency_ratio,monetary_log,Numeric_Mean,recency_x_frequency
count,789.000000,789.000000,789.000000,789.000000,789.000000,789.000000,789.000000,789.000000,789.000000
mean,9.258555,57.328264,5164.568428,107.810324,305926.324778,0.188167,8.361338,44467.691408,553.783270
std,5.517767,24.748019,2701.333852,77.811753,221088.826977,0.134124,0.682005,31892.069529,438.692887
min,1.000000,5.000000,571.450000,6.312600,8613.550000,0.010000,6.349925,1419.881372,10.000000
25%,4.000000,35.000000,2913.900000,52.607065,121102.000000,0.084337,7.977591,17814.185440,182.000000
50%,9.000000,58.000000,4991.520000,88.572540,252038.790000,0.162791,8.515696,36623.284749,438.000000
75%,14.000000,79.000000,7498.840000,138.129804,453440.960000,0.259259,8.922637,65688.597258,832.000000
max,19.000000,99.000000,9999.640000,391.370000,946155.700000,0.593750,9.210404,136576.175548,1862.000000


In [7]:
# Remove missing CustomerID if exists
if "CustomerID" in df.columns:
    df = df.dropna(subset=["CustomerID"])

# Convert InvoiceDate to datetime
if "InvoiceDate" in df.columns:
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Create TotalPrice if not available
if "TotalPrice" not in df.columns:
    if "Quantity" in df.columns and "UnitPrice" in df.columns:
        df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

df.head()

,recency,frequency,monetary,avg_spend_per_purchase,customer_value_score,recency_frequency_ratio,monetary_log,Numeric_Mean,recency_x_frequency
0,5,22,5892.58,256.199130,129636.76,0.217391,8.681619,19403.062592,110
1,13,77,9025.47,115.711154,694961.19,0.166667,9.107917,100600.235105,1001
2,13,71,618.83,8.594861,43936.93,0.180556,6.429445,6379.280695,923
3,3,33,9110.30,267.950000,300639.90,0.088235,9.117271,44294.765072,99
4,15,43,5390.88,122.520000,231807.84,0.340909,8.592649,33912.596223,645


In [11]:
print(df.columns)
# Your dataset already contains RFM columns
rfm = df[["recency", "frequency", "monetary"]].copy()

# Rename to standard format (optional but recommended)
rfm.rename(columns={
    "recency": "Recency",
    "frequency": "Frequency",
    "monetary": "Monetary"
}, inplace=True)

rfm.head()

Index(['recency', 'frequency', 'monetary', 'avg_spend_per_purchase',
       'customer_value_score', 'recency_frequency_ratio', 'monetary_log',
       'Numeric_Mean', 'recency_x_frequency'],
      dtype='object')


,Recency,Frequency,Monetary
0,5,22,5892.58
1,13,77,9025.47
2,13,71,618.83
3,3,33,9110.30
4,15,43,5390.88


In [12]:
rfm.describe()

,Recency,Frequency,Monetary
count,789.000000,789.000000,789.000000
mean,9.258555,57.328264,5164.568428
std,5.517767,24.748019,2701.333852
min,1.000000,5.000000,571.450000
25%,4.000000,35.000000,2913.900000
50%,9.000000,58.000000,4991.520000
75%,14.000000,79.000000,7498.840000
max,19.000000,99.000000,9999.640000


In [13]:
rfm["R_Score"] = pd.qcut(rfm["Recency"], 4, labels=[4,3,2,1])
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 4, labels=[1,2,3,4])
rfm["M_Score"] = pd.qcut(rfm["Monetary"], 4, labels=[1,2,3,4])

rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

rfm.head()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,5,22,5892.58,3,1,3,313
1,13,77,9025.47,2,3,4,234
2,13,71,618.83,2,3,1,231
3,3,33,9110.30,4,1,4,414
4,15,43,5390.88,1,2,3,123


In [14]:
rfm.to_csv("rfm_features.csv", index=False)
print("RFM features saved successfully!")

RFM features saved successfully!


In [15]:
# Make a copy
behavioral_df = df.copy()

# 1️⃣ Purchase Intensity (how frequently customer buys relative to recency)
behavioral_df["purchase_intensity"] = (
    behavioral_df["frequency"] / (behavioral_df["recency"] + 1)
)

# 2️⃣ Spending Intensity (how much customer spends per unit recency)
behavioral_df["spending_intensity"] = (
    behavioral_df["monetary"] / (behavioral_df["recency"] + 1)
)

# 3️⃣ High Value Customer Flag
behavioral_df["high_value_flag"] = (
    behavioral_df["monetary"] > behavioral_df["monetary"].median()
).astype(int)

# 4️⃣ Frequent Buyer Flag
behavioral_df["frequent_buyer_flag"] = (
    behavioral_df["frequency"] > behavioral_df["frequency"].median()
).astype(int)

# 5️⃣ At-Risk Customer Flag (high recency = inactive)
behavioral_df["at_risk_flag"] = (
    behavioral_df["recency"] > behavioral_df["recency"].median()
).astype(int)

behavioral_df.head()

,recency,frequency,monetary,avg_spend_per_purchase,customer_value_score,recency_frequency_ratio,monetary_log,Numeric_Mean,recency_x_frequency,purchase_intensity,spending_intensity,high_value_flag,frequent_buyer_flag,at_risk_flag
0,5,22,5892.58,256.199130,129636.76,0.217391,8.681619,19403.062592,110,3.666667,982.096667,1,0,0
1,13,77,9025.47,115.711154,694961.19,0.166667,9.107917,100600.235105,1001,5.500000,644.676429,1,1,1
2,13,71,618.83,8.594861,43936.93,0.180556,6.429445,6379.280695,923,5.071429,44.202143,0,1,1
3,3,33,9110.30,267.950000,300639.90,0.088235,9.117271,44294.765072,99,8.250000,2277.575000,1,0,0
4,15,43,5390.88,122.520000,231807.84,0.340909,8.592649,33912.596223,645,2.687500,336.930000,1,0,1


In [16]:
# Normalize basic metrics first
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

behavioral_df[["recency_norm",
               "frequency_norm",
               "monetary_norm"]] = scaler.fit_transform(
    behavioral_df[["recency", "frequency", "monetary"]]
)

# Engagement score (higher frequency & monetary, lower recency)
behavioral_df["engagement_score"] = (
    (1 - behavioral_df["recency_norm"]) * 0.4 +
    behavioral_df["frequency_norm"] * 0.3 +
    behavioral_df["monetary_norm"] * 0.3
)

behavioral_df.head()

,recency,frequency,monetary,avg_spend_per_purchase,customer_value_score,recency_frequency_ratio,monetary_log,Numeric_Mean,recency_x_frequency,purchase_intensity,spending_intensity,high_value_flag,frequent_buyer_flag,at_risk_flag,recency_norm,frequency_norm,monetary_norm,engagement_score
0,5,22,5892.58,256.199130,129636.76,0.217391,8.681619,19403.062592,110,3.666667,982.096667,1,0,0,0.222222,0.180851,0.564385,0.534682
1,13,77,9025.47,115.711154,694961.19,0.166667,9.107917,100600.235105,1001,5.500000,644.676429,1,1,1,0.666667,0.765957,0.896675,0.632123
2,13,71,618.83,8.594861,43936.93,0.180556,6.429445,6379.280695,923,5.071429,44.202143,0,1,1,0.666667,0.702128,0.005025,0.345479
3,3,33,9110.30,267.950000,300639.90,0.088235,9.117271,44294.765072,99,8.250000,2277.575000,1,0,0,0.111111,0.297872,0.905672,0.716619
4,15,43,5390.88,122.520000,231807.84,0.340909,8.592649,33912.596223,645,2.687500,336.930000,1,0,1,0.777778,0.404255,0.511172,0.363517


In [17]:
behavioral_df["loyalty_score"] = (
    behavioral_df["frequency"] *
    behavioral_df["monetary"] /
    (behavioral_df["recency"] + 1)
)

behavioral_df.head()

,recency,frequency,monetary,avg_spend_per_purchase,customer_value_score,recency_frequency_ratio,monetary_log,Numeric_Mean,recency_x_frequency,purchase_intensity,spending_intensity,high_value_flag,frequent_buyer_flag,at_risk_flag,recency_norm,frequency_norm,monetary_norm,engagement_score,loyalty_score
0,5,22,5892.58,256.199130,129636.76,0.217391,8.681619,19403.062592,110,3.666667,982.096667,1,0,0,0.222222,0.180851,0.564385,0.534682,21606.126667
1,13,77,9025.47,115.711154,694961.19,0.166667,9.107917,100600.235105,1001,5.500000,644.676429,1,1,1,0.666667,0.765957,0.896675,0.632123,49640.085000
2,13,71,618.83,8.594861,43936.93,0.180556,6.429445,6379.280695,923,5.071429,44.202143,0,1,1,0.666667,0.702128,0.005025,0.345479,3138.352143
3,3,33,9110.30,267.950000,300639.90,0.088235,9.117271,44294.765072,99,8.250000,2277.575000,1,0,0,0.111111,0.297872,0.905672,0.716619,75159.975000
4,15,43,5390.88,122.520000,231807.84,0.340909,8.592649,33912.596223,645,2.687500,336.930000,1,0,1,0.777778,0.404255,0.511172,0.363517,14487.990000


In [18]:
derived_df = df.copy()

# Avoid division by zero
derived_df["frequency_safe"] = derived_df["frequency"] + 1
derived_df["recency_safe"] = derived_df["recency"] + 1
derived_df["monetary_safe"] = derived_df["monetary"] + 1


# 1️⃣ Monetary per Frequency (average basket size)
derived_df["monetary_per_frequency"] = (
    derived_df["monetary_safe"] / derived_df["frequency_safe"]
)

# 2️⃣ Frequency per Recency (activity rate)
derived_df["frequency_per_recency"] = (
    derived_df["frequency_safe"] / derived_df["recency_safe"]
)

# 3️⃣ Monetary per Recency (spend speed)
derived_df["monetary_per_recency"] = (
    derived_df["monetary_safe"] / derived_df["recency_safe"]
)

# 4️⃣ Value Efficiency Ratio
derived_df["value_efficiency_ratio"] = (
    derived_df["customer_value_score"] /
    (derived_df["recency_safe"])
)

# 5️⃣ Spending Consistency Ratio
derived_df["spending_consistency_ratio"] = (
    derived_df["avg_spend_per_purchase"] /
    derived_df["monetary_per_frequency"]
)

derived_df.head()

,recency,frequency,monetary,avg_spend_per_purchase,customer_value_score,recency_frequency_ratio,monetary_log,Numeric_Mean,recency_x_frequency,frequency_safe,recency_safe,monetary_safe,monetary_per_frequency,frequency_per_recency,monetary_per_recency,value_efficiency_ratio,spending_consistency_ratio
0,5,22,5892.58,256.199130,129636.76,0.217391,8.681619,19403.062592,110,23,6,5893.58,256.242609,3.833333,982.263333,21606.126667,0.999830
1,13,77,9025.47,115.711154,694961.19,0.166667,9.107917,100600.235105,1001,78,14,9026.47,115.723974,5.571429,644.747857,49640.085000,0.999889
2,13,71,618.83,8.594861,43936.93,0.180556,6.429445,6379.280695,923,72,14,619.83,8.608750,5.142857,44.273571,3138.352143,0.998387
3,3,33,9110.30,267.950000,300639.90,0.088235,9.117271,44294.765072,99,34,4,9111.30,267.979412,8.500000,2277.825000,75159.975000,0.999890
4,15,43,5390.88,122.520000,231807.84,0.340909,8.592649,33912.596223,645,44,16,5391.88,122.542727,2.750000,336.992500,14487.990000,0.999815


In [19]:
# 6️⃣ Engagement Ratio
derived_df["engagement_ratio"] = (
    (derived_df["frequency"] * derived_df["monetary"]) /
    derived_df["recency_safe"]
)

# 7️⃣ Log Adjusted Spend Ratio
derived_df["log_spend_ratio"] = (
    derived_df["monetary_log"] /
    derived_df["frequency_safe"]
)

# 8️⃣ Growth Potential Ratio
derived_df["growth_potential_ratio"] = (
    derived_df["frequency_per_recency"] *
    derived_df["monetary_per_frequency"]
)

derived_df.head()

,recency,frequency,monetary,avg_spend_per_purchase,customer_value_score,recency_frequency_ratio,monetary_log,Numeric_Mean,recency_x_frequency,frequency_safe,recency_safe,monetary_safe,monetary_per_frequency,frequency_per_recency,monetary_per_recency,value_efficiency_ratio,spending_consistency_ratio,engagement_ratio,log_spend_ratio,growth_potential_ratio
0,5,22,5892.58,256.199130,129636.76,0.217391,8.681619,19403.062592,110,23,6,5893.58,256.242609,3.833333,982.263333,21606.126667,0.999830,21606.126667,0.377462,982.263333
1,13,77,9025.47,115.711154,694961.19,0.166667,9.107917,100600.235105,1001,78,14,9026.47,115.723974,5.571429,644.747857,49640.085000,0.999889,49640.085000,0.116768,644.747857
2,13,71,618.83,8.594861,43936.93,0.180556,6.429445,6379.280695,923,72,14,619.83,8.608750,5.142857,44.273571,3138.352143,0.998387,3138.352143,0.089298,44.273571
3,3,33,9110.30,267.950000,300639.90,0.088235,9.117271,44294.765072,99,34,4,9111.30,267.979412,8.500000,2277.825000,75159.975000,0.999890,75159.975000,0.268155,2277.825000
4,15,43,5390.88,122.520000,231807.84,0.340909,8.592649,33912.596223,645,44,16,5391.88,122.542727,2.750000,336.992500,14487.990000,0.999815,14487.990000,0.195287,336.992500


In [20]:
derived_ratios = derived_df[[
    "monetary_per_frequency",
    "frequency_per_recency",
    "monetary_per_recency",
    "value_efficiency_ratio",
    "spending_consistency_ratio",
    "engagement_ratio",
    "log_spend_ratio",
    "growth_potential_ratio"
]]

derived_ratios.head()

,monetary_per_frequency,frequency_per_recency,monetary_per_recency,value_efficiency_ratio,spending_consistency_ratio,engagement_ratio,log_spend_ratio,growth_potential_ratio
0,256.242609,3.833333,982.263333,21606.126667,0.999830,21606.126667,0.377462,982.263333
1,115.723974,5.571429,644.747857,49640.085000,0.999889,49640.085000,0.116768,644.747857
2,8.608750,5.142857,44.273571,3138.352143,0.998387,3138.352143,0.089298,44.273571
3,267.979412,8.500000,2277.825000,75159.975000,0.999890,75159.975000,0.268155,2277.825000
4,122.542727,2.750000,336.992500,14487.990000,0.999815,14487.990000,0.195287,336.992500


In [21]:
derived_ratios.to_csv("derived_ratio_features.csv", index=False)
print("Derived ratio features saved successfully!")

Derived ratio features saved successfully!
